In [1]:
instances           = {}

import numpy as np
import pandas as pd
import copy
import time
import pickle
from util.util_load          import read_txt
from util.util_display       import plot
from util.data_indentifier   import InstanceData

from env_action.metaheu      import GeneticAlgorithm, random_population
from env_action.action_space import action_space

setting             = 'TIGHT_DUEDATE'
directory           = f"DATA/{setting}_VALIDATION"

planning_horizon    = 480*60
ReworkProbability   = 0.03

PopSize             = 1501
maxtime             = 100
action_name            = ["GA", "TS", "LFOH", "LAPH", "LAP_LFO", 
                        "LFOH-TS", "LAPH-TS", "LFOH-GA", "LAPH-GA",
                        "CDR1", "CDR2", "CDR3", "CDR5", "CDR6",
                        "RCRS"]
action_id           = 3

CaseList            = [f'valid{set+1}' for set in range(15)]

In [2]:
for CaseID in CaseList:
    print(CaseID)
    data_path = f"{directory}/Case{CaseID}_480.txt"
    # data_path = "DATA/jobs_small.txt"
    J, I, K, p_ijk, h_ijk,   \
    d_j, n_j, MC_ji, n_MC_ji,\
    OperationPool          = read_txt(data_path)

    S_k                    = np.zeros((K))
    S_j                    = np.zeros((J))
    n_ops_left_j           = copy.deepcopy(n_j)
    MB_record = {}
    t                      = 0
    JSet                   = list(range(J))
    OJSet                  = [[] for _ in range(J)]
    for j in JSet:
        OJSet[j]           = [i for i in range(int(n_j[j]))]

    T_cur = 0
    Tard_job = 0
    Oij_on_machine = 0
    affected_Oij = 0
    NewJobList = 0
    CT_k = 0
    X_ijk = 0
    S_ij = 0
    C_ij = 0
    C_j = 0
    re = 0

    action_method 						         = action_space(J, I, K, p_ijk, h_ijk, d_j, n_j, 
                                                                MC_ji, n_MC_ji, n_ops_left_j, OperationPool, S_k, S_j, 
                                                                JSet, OJSet, affected_Oij, 
                                                                t, X_ijk, S_ij, C_ij, C_j, CT_k, T_cur, Tard_job,
                                                                NewJobList, PopSize, maxtime, re)
    
    # print(action_name[action_id])

    reschedule							         = action_method[action_id]
    GBest, X_ijk, S_ij, C_ij, C_j                = reschedule()
    # fig1 = plot(J, K, n_j, X_ijk, S_ij, C_ij, MB_record, t)
    # display(fig1)
    remaining_info_file = f'{directory}/Case{CaseID}_{planning_horizon // 60}_InfoNewJob.pkl'
    with open(remaining_info_file, 'rb') as f:
        remaining_batches = pickle.load(f)

    new_job_indices = [comp_id for comp_id, qty in remaining_batches.items() for _ in range(qty)]
    instances[CaseID]             = InstanceData(J, I, X_ijk, S_ij, C_ij, C_j, p_ijk, h_ijk, 
                                                    d_j, n_j, MC_ji, n_MC_ji, OperationPool, new_job_indices)

valid1
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21] [[0, 0], [0, 1], [0, 2], [0, 3], [0, 4], [0, 5], [0, 6], [0, 7], [0, 8], [0, 9], [0, 10], [0, 11], [0, 12], [0, 13], [0, 14], [0, 15], [0, 16], [0, 17], [0, 18], [0, 19], [0, 20], [0, 21]]
0 0 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 1 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 2 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 3 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 4 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 5 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 6 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 7 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 8 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 9 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 10 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [3]:
# Store data
with open(f'{directory}/pickle_valid_instances_480.pkl', 'wb') as f:
    pickle.dump(instances, f)

In [4]:
from util.util_load        import read_scenario
from util.data_indentifier import ScenarioData
import pickle

K = 30
scenarios           = {}
critical_machines   = {5, 6, 7, 8, 9, 10, 11, 12, 13, 21, 22, 26, 27}
ScenarioList        = ['A', 'B', 'C']

SetList = [f'valid{set+1}' for set in range(15)]
for Set in SetList:
    for ScenarioID in ScenarioList:
        scenario_path              = f"{directory}/Scenario{Set}{ScenarioID}_480.txt"
        JA_event, MB_event         = read_scenario(scenario_path, K, critical_machines)
        scenarios[Set+ScenarioID]  = ScenarioData(JA_event, MB_event)

with open(f'{directory}/pickle_valid_scenarios_480.pkl', 'wb') as f:
    pickle.dump(scenarios, f)

In [5]:
# Adjust to benchmark with Luo's DDQN (key reference 2) (since it does not have machine breakdown)

from util.util_load        import read_scenario
from util.data_indentifier import ScenarioData

import pickle
# K = 30
# scenarios           = {}
# critical_machines   = {5, 6, 7, 8, 9, 10, 11, 12, 13, 21, 22, 26, 27}
# ScenarioList        = ['A', 'B', 'C']

# SetList = [f'valid{set+1}' for set in range(15)]
# MB_event = [[] for k in range(K)]
# for Set in SetList:
#     for ScenarioID in ScenarioList:
#         scenario_path = f"{directory}/Scenario{Set}{ScenarioID}_480.txt"
#         JA_event, blank   = read_scenario(scenario_path, K, critical_machines)
#         scenarios[Set+ScenarioID]  = ScenarioData(JA_event, MB_event)

# with open(f'{directory}/pickle_testingJA_valid_scenarios_480.pkl', 'wb') as f:
#     pickle.dump(scenarios, f)

# ========================================================================    

K = 30
scenarios           = {}
critical_machines   = {5, 6, 7, 8, 9, 10, 11, 12, 13, 21, 22, 26, 27}
ScenarioList        = ['A', 'B', 'C']

SetList = [f'valid{set+1}' for set in range(15)]
for Set in SetList:
    for ScenarioID in ScenarioList:
        scenario_path = f"{directory}/Scenario{Set}{ScenarioID}_480.txt"
        JA_event, MB_event     = read_scenario(scenario_path, K, critical_machines)
        real_JA_event = {}
        for job_id, deadline, description in JA_event:
            real_JA_event[job_id] = (deadline, description)
        scenarios[Set+ScenarioID]  = real_JA_event

with open(f'{directory}/pickle_JA_valid_scenarios_480.pkl', 'wb') as f:
    pickle.dump(scenarios, f)